# Recombination nodes with more than two parents

Almost all recombination nodes in the sc2ts ARG have two parents (a single breakpoint), but a
few have more. These are not in `data/recombinants.csv`, because the breakpoint interval and
supporting-loci QC calculations are defined for single breakpoints. This notebook describes them
together, for the supplementary section "Recombination nodes with more than two parents".

Mutation counts with (k=4) and without (k=1000) recombination come from the rematch pipeline
(`arg_postprocessing`, rule `extract_k1000_muts`).

In [1]:
import json
import collections

import numpy as np
import pandas as pd
import tszip
import sc2ts

import nb_utils

In [2]:
ts = tszip.load("../data/sc2ts_viridian_v2.3.trees.tsz")
df_node = sc2ts.node_data(ts, inheritance_stats=False)
re_labels = nb_utils.standard_recombinant_labels(ts)
dfr = pd.read_csv("../data/recombinants.csv")
with open("../arg_postprocessing/sc2ts_v2_2024-06-06_pr_pp_mp_aph_bps_pango_dated_k1000_muts.json") as f:
    rematch_muts = {int(k): v for k, v in json.load(f).items()}

## Identify the multi-parent recombination nodes

In [3]:
recombinants = np.where(ts.nodes_flags & sc2ts.NODE_IS_RECOMBINANT)[0]
num_parents = pd.Series(
    {u: np.unique(ts.edges_parent[ts.edges_child == u]).shape[0] for u in recombinants})
multi_parent = num_parents[num_parents > 2].index.to_numpy()
assert len(recombinants) == 1324
assert set(multi_parent) == set(recombinants) - set(dfr["recombinant"])
assert set(rematch_muts) == set(recombinants)
num_parents.value_counts().sort_index()

2    1320
3       2
4       1
7       1
Name: count, dtype: int64

## Summary of each node

In [4]:
# Descendant samples, taking the maximum over trees as in recombinants.csv
num_descendants = np.zeros(len(multi_parent), dtype=int)
for tree in ts.trees():
    num_descendants = np.maximum(num_descendants, [tree.num_samples(u) for u in multi_parent])

df_multi = pd.DataFrame({
    "label": [re_labels.get(u, "") for u in multi_parent],
    "pango": df_node.loc[multi_parent, "pango"].values,
    "date": df_node.loc[multi_parent, "date"].dt.date.values,
    "num_parents": num_parents[multi_parent].values,
    "num_mutations": df_node.loc[multi_parent, "num_mutations"].values,
    "k4_muts": [rematch_muts[u]["k4_muts"] for u in multi_parent],
    "k1000_muts": [rematch_muts[u]["k1000_muts"] for u in multi_parent],
    "num_descendant_samples": num_descendants,
}, index=pd.Index(multi_parent, name="node"))
df_multi["mutations_averted"] = df_multi.k1000_muts - df_multi.k4_muts
df_multi

,label,pango,date,num_parents,num_mutations,k4_muts,k1000_muts,num_descendant_samples,mutations_averted
node,,,,,,,,,
242319,XB,B.1,2021-03-03,3,20,18,25,426,7
1723601,XAY,XAY.2,2022-12-06,7,17,22,52,371,30
1727022,XBC,XBC.1.2.1,2022-10-09,4,32,31,59,98,28
1854754,,BA.2.86.1,2023-05-24,3,52,47,48,25701,1


`num_mutations` is the number of mutations on the node in the final ARG, which differs from the
k=4 match count because of post-processing (e.g. mutation parsimony).

## Parent segments

One row per edge, left to right along the genome. A parent is "exact" if it is a sample, or has
a sample child with no mutations (as for boldface parents in the Pango X table).

In [5]:
def is_exact(parent):
    if df_node.loc[parent, "is_sample"]:
        return True
    children = ts.edges_child[ts.edges_parent == parent]
    df = df_node.loc[children]
    return bool(np.any(df["is_sample"] & (df["num_mutations"] == 0)))

rows = []
for u in multi_parent:
    edges = np.where(ts.edges_child == u)[0]
    for e in edges[np.argsort(ts.edges_left[edges])]:
        p = ts.edges_parent[e]
        rows.append({
            "node": u,
            "label": re_labels.get(u, ""),
            "left": int(ts.edges_left[e]),
            "right": int(ts.edges_right[e]),
            "parent": p,
            "parent_pango": df_node.loc[p, "pango"],
            "parent_date": df_node.loc[p, "date"].date(),
            "parent_exact": is_exact(p),
        })
df_segments = pd.DataFrame(rows)
df_segments

,node,label,left,right,parent,parent_pango,parent_date,parent_exact
0,242319,XB,0,6633,8333,B.1,2020-04-09,True
1,242319,XB,6633,23012,182652,B.1.634,2021-03-03,True
2,242319,XB,23012,29904,207670,B.1,2020-09-03,False
3,1723601,XAY,0,4456,1318093,BA.2,2022-03-03,True
4,1723601,XAY,4456,10198,680288,AY.46,2021-10-04,True
5,1723601,XAY,10198,15714,1718647,XBB.1.5.21,2022-12-06,True
6,1723601,XAY,15714,22118,319506,AY.4,2021-06-13,True
7,1723601,XAY,22118,25000,1667877,BA.5.2.1,2022-09-23,True
8,1723601,XAY,25000,27575,401684,AY.4,2021-07-26,True
9,1723601,XAY,27575,29904,1519541,BA.2,2022-06-11,True


In [6]:
# Parent lineages left to right, with breakpoints, for the supplementary table
for u, df in df_segments.groupby("node"):
    parts = [df.iloc[0]["parent_pango"]]
    for _, row in df.iloc[1:].iterrows():
        parts += [str(row["left"]), row["parent_pango"]]
    print(u, re_labels.get(u, ""), " | ".join(parts))

242319 XB B.1 | 6633 | B.1.634 | 23012 | B.1
1723601 XAY BA.2 | 4456 | AY.46 | 10198 | XBB.1.5.21 | 15714 | AY.4 | 22118 | BA.5.2.1 | 25000 | AY.4 | 27575 | BA.2
1727022 XBC BA.2 | 4184 | AY.38 | 22674 | BM.1.1 | 25584 | AY.9
1854754  BA.2 | 22898 | CM.8.1 | 24990 | BA.2


## Descendant samples by Pango lineage

In [7]:
# Use the tree in which each node has the most descendant samples
descendant_pango = {}
for u in multi_parent:
    index = np.argmax([tree.num_samples(u) for tree in ts.trees()])
    tree = ts.at_index(index)
    counts = collections.Counter(df_node.loc[list(tree.samples(u)), "pango"])
    descendant_pango[u] = dict(counts.most_common(8))
descendant_pango

{np.int64(242319): {'B.1': 224, 'XB': 202},
 np.int64(1723601): {'XAY.2.3': 127,
  'XAY.2': 91,
  'XAY.2.1': 81,
  'XAY.2.2': 72},
 np.int64(1727022): {'XBC.1.2.1': 98},
 np.int64(1854754): {'JN.1': 7448,
  'JN.1.4': 4741,
  'JN.1.1': 1571,
  'JN.1.4.5': 1030,
  'BA.2.86.1': 656,
  'JN.1.8.1': 586,
  'JN.1.7': 570,
  'JN.1.39': 452}}

## XB parent lineages

XB was designated as a recombinant of B.1.634 and B.1.631. Both lineages have samples in the ARG.

In [8]:
df_sample = df_node[df_node["is_sample"]]
df_sample["pango"].isin(["B.1.634", "B.1.631"]).groupby(df_sample["pango"]).sum().loc[["B.1.634", "B.1.631"]]

pango
B.1.634    59
B.1.631    52
Name: pango, dtype: int64

## BA.2.86

Node 1854754 is the origin of BA.2.86.1 in the ARG, and all BA.2.86 and JN.1 samples descend
from it. It sits on a long branch, and allowing recombination saves only a single mutation, so
the recombination is likely spurious.

In [9]:
df_pe = pd.read_csv("../data/pango_events_in_arg.csv")
df_pe[df_pe["pango"].str.startswith("BA.2.86")][["pango", "root", "total_samples", "max_descendants"]]

,pango,root,total_samples,max_descendants
1153,BA.2.86,1856480,14,13
1154,BA.2.86.1,1854754,656,656
1155,BA.2.86.2,1865319,16,7
1156,BA.2.86.3,1862472,11,11
1157,BA.2.86.4,1859482,1,1
1158,BA.2.86.6,1862213,2,2
1159,BA.2.86.7,1857192,1,1


In [10]:
u = 1854754
index = np.argmax([tree.num_samples(u) for tree in ts.trees()])
descendants = set(ts.at_index(index).samples(u))
df_sample = df_node[df_node["is_sample"]]
for prefix in ["BA.2.86", "JN.1"]:
    ids = df_sample.index[df_sample["pango"].str.startswith(prefix)]
    print(prefix, len(ids), "samples,", len(set(ids) & descendants), "descend from", u)

BA.2.86 701 samples, 701 descend from 1854754


JN.1 21908 samples, 21908 descend from 1854754
